# 🗂️ Triage Model — *what is this, and how urgent?*

Free-text incident descriptions are embedded with **LaBSE** (multilingual, so
English *and* Kannada/Hindi share one space), then a **soft-vote ensemble**
(LightGBM + LogisticRegression, both class-weighted) predicts two heads:

* 🏷️ **cause** — 16-class (accident, water_logging, …)
* 🚦 **priority** — binary (High / Low)

> The ensemble averages the two classifiers' probabilities — it beats either
> alone (cause macro-F1 0.481 → 0.494, priority 0.668 → 0.673).

This **retrains and overwrites** `models/triage/`. First run downloads LaBSE
(~1.8 GB) and embeds all rows; later runs reuse the cached embeddings.

In [ ]:
# 📦 Setup — locate the backend/ root (folder containing app/ and models/)
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "app").is_dir() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))
print("✅ Environment ready")
print(f"   📂 Working dir : {os.getcwd()}")
print(f"   🐍 Python      : {sys.version.split()[0]}")

## 📦 Imports & configuration

In [ ]:
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from app.config import settings
from app.engines.triage.ensemble import SoftVoteEnsemble
from app.features.text import LABSE_MODEL, TextEmbedder, clean_text, detect_lang

TEST_FRACTION = 0.20
RANDOM_SEED   = 42
_MODEL_DIR    = settings.models_dir / "triage"
_CACHE_FILE   = settings.interim_dir / "triage_embeddings.npz"

print("✅ Config loaded")
print(f"   🌐 Embedder: {LABSE_MODEL}")
print(f"   🤖 Heads   : SoftVote(LightGBM + LogReg, 0.5/0.5, class_weight='balanced')")

## 📥 Step 1 — Rows with usable text + labels

In [ ]:
from app.ml.pipeline import load_clean

print("📥 Loading events and selecting rows with description + labels ...")
df = load_clean().assign(_text=lambda d: d["description"].map(clean_text))
mask = (df["_text"].str.len() > 0) & df["cause_norm"].notna() & df["priority"].notna()
rows = (df.loc[mask, ["id", "_text", "cause_norm", "priority"]]
          .sort_values("id", kind="mergesort").reset_index(drop=True))

print(f"✅ {len(rows):,} usable rows")
print(f"   🏷️  {rows['cause_norm'].nunique()} cause classes  ·  🚦 priority split:")
print(rows["priority"].value_counts().to_string())
rows.head(3)

## 🌐 Step 2 — Embed descriptions with LaBSE (cached)

In [ ]:
ids = rows["id"].to_numpy()
emb = None
if _CACHE_FILE.exists():
    cached = np.load(_CACHE_FILE, allow_pickle=True)
    if np.array_equal(cached["ids"], ids):
        emb = cached["emb"].astype(np.float32)
        print(f"♻️  Reusing cached embeddings  ({_CACHE_FILE.name})")

if emb is None:
    print("🌐 Encoding with LaBSE (first run downloads the model) ...")
    embedder = TextEmbedder(LABSE_MODEL); embedder.load()
    emb = embedder.encode(rows["_text"].tolist())
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(_CACHE_FILE, ids=ids, emb=emb)
    print("💾 Cached embeddings for next time")

print(f"✅ Embedding matrix: {emb.shape[0]:,} × {emb.shape[1]}")

## 🔀 Step 3 — Encode labels & stratified split

In [ ]:
cause_le    = LabelEncoder().fit(rows["cause_norm"])
priority_le = LabelEncoder().fit(rows["priority"])
y_cause     = cause_le.transform(rows["cause_norm"])
y_priority  = priority_le.transform(rows["priority"])
is_indic    = rows["_text"].map(lambda t: detect_lang(t) == "indic").to_numpy()

train_idx, test_idx = train_test_split(
    np.arange(len(rows)), test_size=TEST_FRACTION, random_state=RANDOM_SEED, stratify=y_cause)

print("🔀 Stratified 80/20 split")
print(f"   🟦 train: {len(train_idx):,}   🟧 test: {len(test_idx):,}")
print(f"   🌏 Indic (Kannada/Hindi) rows in test: {int(is_indic[test_idx].sum())}")
print(f"   🏷️  classes: {list(cause_le.classes_)}")

## 🤖 Step 4 — Train the soft-vote ensemble heads

In [ ]:
def fit_head(x, y):
    """Fit LightGBM + LogReg and average their probabilities (50/50)."""
    lgb = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                         class_weight="balanced", random_state=RANDOM_SEED,
                         n_jobs=-1, verbose=-1).fit(x, y)
    lr = LogisticRegression(max_iter=2000, class_weight="balanced",
                            C=1.0, random_state=RANDOM_SEED).fit(x, y)
    return SoftVoteEnsemble([lgb, lr], weights=[0.5, 0.5])

print("🤖 Training soft-vote ensemble heads (LightGBM + LogReg) ...")
print("   → cause head (16-class) ...", end=" ")
cause_head = fit_head(emb[train_idx], y_cause[train_idx]); print("✅")
print("   → priority head (binary) ...", end=" ")
priority_head = fit_head(emb[train_idx], y_priority[train_idx]); print("✅")
print("🎉 Both ensemble heads trained")

## 📊 Step 5 — Evaluate with macro-F1 (overall + Indic subset)

In [ ]:
def macro_f1(yt, yp):
    return round(float(f1_score(yt, yp, average="macro", zero_division=0)), 4)

cause_pred    = cause_head.predict(emb[test_idx])
priority_pred = priority_head.predict(emb[test_idx])
yt_c, yt_p    = y_cause[test_idx], y_priority[test_idx]
it            = is_indic[test_idx]

metrics = {
    "cause_macro_f1": macro_f1(yt_c, cause_pred),
    "cause_accuracy": round(float(accuracy_score(yt_c, cause_pred)), 4),
    "priority_macro_f1": macro_f1(yt_p, priority_pred),
    "priority_accuracy": round(float(accuracy_score(yt_p, priority_pred)), 4),
    "test_n": int(len(test_idx)),
    "indic_test_n": int(it.sum()),
}
if it.any():
    metrics["cause_macro_f1_indic"] = macro_f1(yt_c[it], cause_pred[it])
    metrics["priority_macro_f1_indic"] = macro_f1(yt_p[it], priority_pred[it])

print("📊 Held-out performance (macro-F1 is the honest metric under imbalance)")
print(f"   🏷️  cause    macro-F1 {metrics['cause_macro_f1']:.3f}  ·  acc {metrics['cause_accuracy']:.3f}")
print(f"   🚦 priority macro-F1 {metrics['priority_macro_f1']:.3f}  ·  acc {metrics['priority_accuracy']:.3f}")
print(f"   🌏 Indic   cause {metrics.get('cause_macro_f1_indic','-')}  ·  "
      f"priority {metrics.get('priority_macro_f1_indic','-')}")
metrics

## 💾 Step 6 — Persist heads, encoders & registry

In [ ]:
_MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(cause_head, _MODEL_DIR / "cause_head.joblib")
joblib.dump(priority_head, _MODEL_DIR / "priority_head.joblib")
joblib.dump({"cause": cause_le, "priority": priority_le}, _MODEL_DIR / "label_encoders.joblib")

entry = {
    "model": "triage",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "embedder": LABSE_MODEL,
    "embedding_dim": int(emb.shape[1]),
    "head_estimator": "SoftVote(LGBM+LogReg, 0.5/0.5, class_weight=balanced)",
    "heads": {"cause": "multiclass", "priority": "binary"},
    "n_classes": int(len(cause_le.classes_)),
    "n_train": int(len(train_idx)),
    "split": {"type": "stratified", "test_fraction": TEST_FRACTION},
    "classes": cause_le.classes_.tolist(),
    "metrics": metrics,
    "artifacts": {
        "cause_head": "triage/cause_head.joblib",
        "priority_head": "triage/priority_head.joblib",
        "label_encoders": "triage/label_encoders.joblib",
    },
}
registry_path = settings.models_dir / "registry.json"
registry = json.loads(registry_path.read_text()) if registry_path.exists() else {}
registry["triage"] = entry
registry_path.write_text(json.dumps(registry, indent=2))

print("💾 Saved heads + encoders + registry")
print("🏁 Triage training complete!")